# ĐÁNH GIÁ VÀ KIỂM ĐỊNH VĨ MÔ CẤP ĐỘ 1 (LEVEL 1 MACRO FLOW EVALUATION)
## Môi trường Kiểm chuẩn Vi mô SUMO Hiệu chuẩn theo Vận tốc Thực nghiệm (Speed-Calibrated Heterogeneous Sublane Testbed)

> **Dự án:** Điều khiển Đèn Giao thông Thích ứng bằng Học tăng cường Đa tác tử (MARL-TSC)  
> **Khu vực áp dụng:** Trung tâm Quận 1 & Quận 3, TP. Hồ Chí Minh (35 nút tín hiệu, 66 điểm quan trắc TomTom)  
> **Tham chiếu lý thuyết:** `macro_traffic_crawler/docs/SD_EVALUATION.md`

---
### Mục tiêu Notebook:
1. Tải và phân tích tệp `simulation_edgedata.xml` trích xuất từ phần mềm mô phỏng vi mô SUMO.
2. Tích hợp dữ liệu quan trắc thực tế từ TomTom API (bao gồm khảo sát độ phủ EDA và trọng số $w_e = 1/\hat{\sigma}^2(c)$).
3. Đo lường toàn diện **Bộ 4 Trụ Cột Động học Vận tốc Độc lập**:
   - **Trụ cột 1:** Sai số vận tốc tuyệt đối & tương đối (RMSE, MAE, MRE so với $v_0$).
   - **Trụ cột 2:** Tương quan thứ hạng tắc nghẽn (Spearman Rank $\rho$, Kendall $\tau$).
   - **Trụ cột 3:** Độ tương đồng hình học phân bố vận tốc (Wasserstein $W_1$, KS Statistic $D$).
   - **Trụ cột 4:** Ma trận nhầm lẫn phân loại mức phục vụ LOS ($C, S, F$, Cohen's Kappa $\kappa$).
4. Đối chuẩn khoa học với **3 Mốc Tham chiếu** (Free-flow $v_0$, Historical Median, Sàn nhiễu lý thuyết $\sigma_{\text{floor}} = \Delta_{\text{noise}}/\sqrt{2}$).
5. Kiểm định thống kê **Spatial Cluster Bootstrap theo Hành lang** và xuất bảng mã nguồn LaTeX cho khóa luận.

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm đường dẫn thư mục hiện tại vào sys.path
CURRENT_DIR = Path(".").resolve()
if str(CURRENT_DIR) not in sys.path:
    sys.path.insert(0, str(CURRENT_DIR))

from speed_metrics import evaluate_speed_dynamics_suite, classify_los
from baseline_engine import load_multiday_traffic_data, compute_lodo_benchmark_anchors, benchmark_simulation_performance
from statistical_tester import spatial_cluster_bootstrap, lodo_paired_test, hierarchical_two_tier_ci
from evaluate_level1_macro import parse_sumo_edgedata, align_and_prepare_evaluation_data, run_level1_evaluation

print("[+] Đã nạp thành công toàn bộ hệ thống thư viện kiểm chuẩn!")

## 1. Thực thi Kiểm định Vĩ mô Cấp độ 1
Chạy quy trình kiểm chuẩn hoàn chỉnh trên kịch bản SUMO mục tiêu và ngày kiểm định TomTom.

In [ ]:
# Cấu hình đường dẫn dữ liệu kiểm nghiệm
EDGEDATA_PATH = "../../sumo_scenarios/q1_20261020_6h/simulation_edgedata_20261020_6h.xml"
BENCHMARK_GLOB = "../../macro_traffic_crawler/data/processed/*.csv"
TEST_DATE = "2026-10-20"
OUTPUT_DIR = "reports"

results = run_level1_evaluation(
    edgedata_path=EDGEDATA_PATH,
    benchmark_source=BENCHMARK_GLOB,
    output_dir=OUTPUT_DIR,
    test_date=TEST_DATE,
    min_confidence_floor=0.40
)

## 2. Bảng Tổng kết Số liệu Bộ 4 Trụ Cột Động học Vận tốc

In [ ]:
p1 = results["pillar1_speed_errors"]
p2 = results["pillar2_rank_correlation"]
p3 = results["pillar3_distribution_divergence"]
p4 = results["pillar4_los_regimes"]
validity = results["scientific_validity"]

summary_table = pd.DataFrame([
    {"Trụ cột": "Trụ cột 1 - Sai số Vận tốc", "Chỉ số": "RMSE", "Giá trị": f"{p1['rmse']} km/h", "Mục tiêu / Tiêu chuẩn": "≤ Historical Median"},
    {"Trụ cột": "Trụ cột 1 - Sai số Vận tốc", "Chỉ số": "MAE", "Giá trị": f"{p1['mae']} km/h", "Mục tiêu / Tiêu chuẩn": "Sai số tuyệt đối trung bình"},
    {"Trụ cột": "Trụ cột 1 - Sai số Vận tốc", "Chỉ số": "Bias (Mean Error)", "Giá trị": f"{p1['bias']} km/h", "Mục tiêu / Tiêu chuẩn": "Tiệm cận 0 (không lệch hệ thống)"},
    {"Trụ cột": "Trụ cột 1 - Sai số Vận tốc", "Chỉ số": "MRE so với v0", "Giá trị": f"{p1['mre']}%", "Mục tiêu / Tiêu chuẩn": "< 20%"},
    {"Trụ cột": "Trụ cột 2 - Tương quan Hạng", "Chỉ số": "Spearman Rank ρ", "Giá trị": f"{p2['spearman_rho']:.4f}", "Mục tiêu / Tiêu chuẩn": "ρ ≥ 0.70, p < 0.05"},
    {"Trụ cột": "Trụ cột 2 - Tương quan Hạng", "Chỉ số": "Kendall Tau τ", "Giá trị": f"{p2['kendall_tau']:.4f}", "Mục tiêu / Tiêu chuẩn": "Đồng thuận xếp hạng"},
    {"Trụ cột": "Trụ cột 3 - Hình dáng Phân bố", "Chỉ số": "Wasserstein Distance W1", "Giá trị": f"{p3['wasserstein_w1']} km/h", "Mục tiêu / Tiêu chuẩn": "Tiệm cận 0 km/h"},
    {"Trụ cột": "Trụ cột 3 - Hình dáng Phân bố", "Chỉ số": "KS Distance D", "Giá trị": f"{p3['ks_statistic_d']:.4f}", "Mục tiêu / Tiêu chuẩn": "Thước đo mô tả hình học"},
    {"Trụ cột": "Trụ cột 4 - Phân loại LOS", "Chỉ số": "Accuracy", "Giá trị": f"{p4['accuracy']*100:.1f}%", "Mục tiêu / Tiêu chuẩn": "Độ chính xác phân loại 3 mức"},
    {"Trụ cột": "Trụ cột 4 - Phân loại LOS", "Chỉ số": "Macro F1-Score", "Giá trị": f"{p4['macro_f1']:.4f}", "Mục tiêu / Tiêu chuẩn": "Cân bằng giữa các mức LOS"},
    {"Trụ cột": "Trụ cột 4 - Phân loại LOS", "Chỉ số": "Cohen's Kappa κ", "Giá trị": f"{p4['cohen_kappa']:.4f}", "Mục tiêu / Tiêu chuẩn": "κ > 0.40 (loại trừ ngẫu nhiên)"},
])

summary_table

## 3. Trực quan hóa Ma trận Nhầm lẫn Phân loại Mức Dịch vụ (LOS Confusion Matrix)

In [ ]:
cm_matrix = np.array(p4["matrix_array"])
los_labels = ["Congested (C)", "Slow (S)", "Free-flow (F)"]

plt.figure(figsize=(7, 5), dpi=150)
sns.heatmap(
    cm_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=los_labels,
    yticklabels=los_labels,
    cbar=False,
    linewidths=1.0,
    linecolor="black"
)
plt.title(f"LOS Regime Classification Matrix\nMacro F1: {p4['macro_f1']:.3f} | Cohen's Kappa: {p4['cohen_kappa']:.3f}", fontweight="bold")
plt.xlabel("SUMO Simulated Regime", fontweight="bold")
plt.ylabel("TomTom Observed Ground-Truth", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Đối chuẩn Baselines & Phân định Vùng Chuẩn Khoa học (Scientific Validity Zone)

In [ ]:
bench = validity["benchmarks"]
zone = validity.get("acceptable_zone_range", [0, 0])

print("=" * 65)
print(f"[*] TRẠNG THÁI KHOA HỌC: {validity['status']}")
print(f"[*] ĐÁNH GIÁ:            {validity['message']}")
print("=" * 65)
print(f"    • Mốc 1 - Free-flow RMSE (v0):        {bench['freeflow_rmse']:.3f} km/h")
print(f"    • Mốc 2 - Historical Median RMSE:     {bench['historical_median_rmse']:.3f} km/h")
print(f"    • Mốc 3 - Sàn nhiễu lý thuyết:        {bench['theoretical_noise_floor']:.3f} km/h")
print(f"    • Vùng chuẩn khoa học:                [{zone[0]:.3f}, {zone[1]:.3f}] km/h")
print(f"    • Sai số SUMO đạt được:               {validity['sumo_rmse']:.3f} km/h")
print("=" * 65)

## 5. Xuất Bảng Mã nguồn LaTeX cho Luận văn (Overleaf Ready)

In [ ]:
latex_table = f"""\\begin{{table}}[htbp]
\\centering
\\caption{{Bảng Đánh giá Động học Vận tốc Cấp độ 1 cho Môi trường Mô phỏng Vi mô SUMO}}
\\label{{tab:level1_macro_evaluation}}
\\begin{{tabular}}{{llcc}}
\\hline
\\textbf{{Trụ cột Đánh giá}} & \\textbf{{Chỉ số}} & \\textbf{{Giá trị Đạt được}} & \\textbf{{Ngưỡng Chuẩn / Baseline}} \\\\
\\hline
\\multirow{{3}}{{*}}{{1. Sai số Vận tốc}} & RMSE & {p1['rmse']:.2f} km/h & $\\le {bench['historical_median_rmse']:.2f}$ km/h (Historical Median) \\\\
 & MAE & {p1['mae']:.2f} km/h & - \\\\
 & MRE (theo $v_0$) & {p1['mre']:.1f}\\% & $< 20\\%$ \\\\
\\hline
\\multirow{{2}}{{*}}{{2. Tương quan Hạng}} & Spearman $\\rho$ & {p2['spearman_rho']:.3f} & $\\ge 0.70$ ($p < 0.05$) \\\\
 & Kendall $\\tau$ & {p2['kendall_tau']:.3f} & - \\\\
\\hline
\\multirow{{2}}{{*}}{{3. Phân bố Vận tốc}} & Wasserstein $W_1$ & {p3['wasserstein_w1']:.2f} km/h & Tiệm cận 0 \\\\
 & KS Distance $D$ & {p3['ks_statistic_d']:.3f} & Thước đo mô tả hình học \\\\
\\hline
\\multirow{{3}}{{*}}{{4. Phân loại LOS}} & Accuracy & {p4['accuracy']*100:.1f}\\% & - \\\\
 & Macro F1-Score & {p4['macro_f1']:.3f} & Cân bằng các trạng thái \\\\
 & Cohen's Kappa $\\kappa$ & {p4['cohen_kappa']:.3f} & $> 0.40$ (Đồng thuận thực chất) \\\\
\\hline
\\multicolumn{{2}}{{l}}{{\\textbf{{Sàn nhiễu Lý thuyết}}}} & \\multicolumn{{2}}{{c}}{{$\\sigma_{{\\text{{floor}}}} = {bench['theoretical_noise_floor']:.2f}$ km/h (Ngưỡng chặn dưới tối ưu)}} \\\\
\\hline
\\end{{tabular}}
\\end{{table}}"""

print(latex_table)